<a href="https://colab.research.google.com/github/RonazAbbasi/ml-zoomcamp2026/blob/main/Homework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

df = pd.read_csv(url)

features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year'
]

target = 'fuel_efficiency_mpg'

print("Q1 - Missing values:")
print(df.isnull().sum())

print()

print("Q2 - Median horsepower:")
print(df['horsepower'].median())

print()


def split_data(df, seed):
    n = len(df)

    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)

    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    return df_train, df_val, df_test


def train_linear_regression(X, y):
    X = np.column_stack([
        np.ones(X.shape[0]),
        X
    ])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


def train_linear_regression_reg(X, y, r=0):
    X = np.column_stack([
        np.ones(X.shape[0]),
        X
    ])

    XTX = X.T.dot(X)

    reg = r * np.eye(XTX.shape[0])
    reg[0, 0] = 0

    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]


def rmse(y, y_pred):
    error = y - y_pred
    mse = (error ** 2).mean()

    return np.sqrt(mse)


# Q3

df_train, df_val, df_test = split_data(df, 42)

df_train_0 = df_train.copy()
df_val_0 = df_val.copy()

df_train_0[features] = df_train_0[features].fillna(0)
df_val_0[features] = df_val_0[features].fillna(0)

X_train = df_train_0[features].values
y_train = df_train_0[target].values

X_val = df_val_0[features].values
y_val = df_val_0[target].values

w0, w = train_linear_regression(X_train, y_train)

y_pred = w0 + X_val.dot(w)

rmse_0 = rmse(y_val, y_pred)


df_train_mean = df_train.copy()
df_val_mean = df_val.copy()

mean_horsepower = df_train_mean['horsepower'].mean()

df_train_mean['horsepower'] = df_train_mean['horsepower'].fillna(mean_horsepower)
df_val_mean['horsepower'] = df_val_mean['horsepower'].fillna(mean_horsepower)

X_train = df_train_mean[features].values
y_train = df_train_mean[target].values

X_val = df_val_mean[features].values
y_val = df_val_mean[target].values

w0, w = train_linear_regression(X_train, y_train)

y_pred = w0 + X_val.dot(w)

rmse_mean = rmse(y_val, y_pred)

print("Q3")
print("RMSE with 0:", round(rmse_0, 3))
print("RMSE with mean:", round(rmse_mean, 3))

if round(rmse_0, 3) < round(rmse_mean, 3):
    print("Answer: With 0")
elif round(rmse_mean, 3) < round(rmse_0, 3):
    print("Answer: With mean")
else:
    print("Answer: Both are equally good")

print()


# Q4

df_train, df_val, df_test = split_data(df, 42)

df_train = df_train.copy()
df_val = df_val.copy()

df_train[features] = df_train[features].fillna(0)
df_val[features] = df_val[features].fillna(0)

X_train = df_train[features].values
y_train = df_train[target].values

X_val = df_val[features].values
y_val = df_val[target].values

r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

scores = {}

print("Q4")

for r in r_values:
    w0, w = train_linear_regression_reg(
        X_train,
        y_train,
        r
    )

    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    scores[r] = score

    print("r =", r, "RMSE =", round(score, 4))

best_r = min(
    r_values,
    key=lambda r: (round(scores[r], 4), r)
)

print("Answer:", best_r)

print()


# Q5

scores = []

print("Q5")

for seed in range(10):

    df_train, df_val, df_test = split_data(df, seed)

    df_train = df_train.copy()
    df_val = df_val.copy()

    df_train[features] = df_train[features].fillna(0)
    df_val[features] = df_val[features].fillna(0)

    X_train = df_train[features].values
    y_train = df_train[target].values

    X_val = df_val[features].values
    y_val = df_val[target].values

    w0, w = train_linear_regression(
        X_train,
        y_train
    )

    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    scores.append(score)

    print("seed =", seed, "RMSE =", round(score, 3))

std = np.std(scores)

print("Standard deviation:", round(std, 3))

print()


# Q6

df_train, df_val, df_test = split_data(df, 9)

df_train_full = pd.concat([
    df_train,
    df_val
])

df_train_full = df_train_full.copy()
df_test = df_test.copy()

df_train_full[features] = df_train_full[features].fillna(0)
df_test[features] = df_test[features].fillna(0)

X_train = df_train_full[features].values
y_train = df_train_full[target].values

X_test = df_test[features].values
y_test = df_test[target].values

w0, w = train_linear_regression_reg(
    X_train,
    y_train,
    r=0.001
)

y_pred = w0 + X_test.dot(w)

test_score = rmse(y_test, y_pred)

print("Q6")
print("Test RMSE:", round(test_score, 3))

Q1 - Missing values:
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

Q2 - Median horsepower:
254.0

Q3
RMSE with 0: 2.205
RMSE with mean: 2.202
Answer: With mean

Q4
r = 0 RMSE = 2.2053
r = 0.01 RMSE = 2.2053
r = 0.1 RMSE = 2.2053
r = 1 RMSE = 2.2053
r = 5 RMSE = 2.2053
r = 10 RMSE = 2.2053
r = 100 RMSE = 2.2053
Answer: 0

Q5
seed = 0 RMSE = 2.239
seed = 1 RMSE = 2.202
seed = 2 RMSE = 2.163
seed = 3 RMSE = 2.204
seed = 4 RMSE = 2.202
seed = 5 RMSE = 2.246
seed = 6 RMSE = 2.27
seed = 7 RMSE = 2.191
seed = 8 RMSE = 2.217
seed = 9 RMSE = 2.207
Standard deviation: 0.029

Q6
Test RMSE: 2.236
